# 02 — Exploração dos resultados

Este notebook lê artefatos imutáveis de `results/runs/`. Ele não treina modelos, não seleciona configurações e não acessa automaticamente o teste final.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RUNS_DIR = ROOT / 'results' / 'runs'
runs = sorted(path for path in RUNS_DIR.iterdir() if path.is_dir() and (path / 'manifest.json').is_file())
print(f'Runs registrados: {len(runs)}')
[path.name for path in runs]

## Comparação de retrieval nos splits internos

In [ ]:
rows = []
for run in runs:
    manifest = json.loads((run / 'manifest.json').read_text(encoding='utf-8'))
    assert manifest['final_test_used'] is False, f'Run final bloqueado no notebook: {run.name}'
    config = json.loads((run / 'config.json').read_text(encoding='utf-8'))
    metrics = json.loads((run / 'metrics.json').read_text(encoding='utf-8'))
    for split, values in metrics.items():
        for k in [1, 3, 5, 10]:
            key = f'answerable_context_recall_at_{k}'
            if key in values:
                rows.append({'run_id': run.name, 'método': config['method'], 'split': split, 'k': k, 'recall_contexto': values[key]})
retrieval_df = pd.DataFrame(rows, columns=['run_id', 'método', 'split', 'k', 'recall_contexto'])
retrieval_df

In [ ]:
selection = retrieval_df[retrieval_df['split'] == 'selection']
if selection.empty:
    print('Ainda não há runs de retrieval por chunks.')
else:
    ax = selection.pivot(index='k', columns='método', values='recall_contexto').plot(marker='o', figsize=(8, 4))
    ax.set(title='Recall de contexto — selection interno', xlabel='k chunks', ylabel='Recall', ylim=(0, 1))
    ax.legend(title='Método', frameon=False)
    plt.tight_layout()

As visualizações exploratórias não substituem os scripts de geração das figuras finais. Toda figura destinada ao TCC deverá registrar o `run_id`.